# Backend contracts - configure, deploy, and validate
Inspect the backend configuration exported by the selected azd environment, deploy
the existing onboarding contract, and verify routing using real HTTP and SDK calls.
Requires subscription deployment permissions and APIM access. This changes shared
backend routing: confirm the complete effective configuration before deploying.
No model is provisioned merely because it appears in an example.

All functionality tests use **Universal LLM** (`/models`) only: model/deployment discovery, chat, embeddings, Responses creation (POST only), SDK streaming, and aliases. The temporary access contract grants only `universal-llm-api`; other API deployments are not changed.


## 1. Configure

### 1. Configure

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming` when that scenario is selected. This notebook tests Responses **creation only** (POST `/models/responses`); GET response and GET input-items are not run. The All Models notebook retains retrieval coverage.



In [3]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(root / "shared"))
import utils
with utils.notebook_step('1. Configure', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    from pathlib import Path
    import sys, json, time, uuid
    import requests
    from validationtools import ValidationContext, ValidationRun
    from contracttools import write_parameters, deploy_contract, access_policy
    from apimtools import APIMClientTool
    from contracttools import scoped_deployment_name
    from consumerhelpers import product_exists

    environment = None  # None uses the selected azd environment.
    from_azd = True
    overrides = {}  # Explicit resource coordinates/model capabilities win over azd.
    # Optional SDK chat-model pin; None selects an eligible model. Example: "my-chat-model".
    model_override = None
    cleanup_enabled = False

    scenario = "backend-onboarding"
    # None loads MODEL_ALIASES/LLM_MODEL_ALIASES from azd; [] explicitly deploys no aliases.
    # Example: [{"name": "chat-alias", "models": ["my-chat-model"], "strategy": "priority"}]
    model_aliases = None

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None



## 2. Initialize

### 2. Initialize

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [4]:
with utils.notebook_step('2. Initialize', inputs={'scenario': globals().get('scenario'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'from_azd': globals().get('from_azd')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    mandatory_inputs.update({
        "onboarding.llm_backends": context.config.get("llm_backends"),
    })
    utils.require_inputs(mandatory_inputs)
    utils.require_capabilities(context.catalog)
    catalog = context.catalog
    if model_override is not None:
        catalog.select("chat", override=model_override)
    if model_aliases is None:
        model_aliases = context.config.get("model_aliases", [])
    apim = APIMClientTool(context.config["resource_group"], context.config.get("apim_name") or "")
    apim.initialize_context(context.account)
    location = context.config["location"]
    run_id = uuid.uuid4().hex[:12]
    run_dir = root / "validation" / ".runs" / scenario / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    run = ValidationRun(scenario)
    apim_coordinates = {"subscriptionId": context.config["subscription_id"],
                        "resourceGroupName": context.config["resource_group"], "name": apim.apim_resource_name}
    utils.format_output("Resolved deployment", {
        "environment": context.config.get("environment") or environment or "selected azd environment",
        "subscription ID": context.config["subscription_id"],
        "subscription name": context.account.get("name"),
        "tenant ID": context.config.get("tenant_id") or context.account.get("tenantId"),
        "resource group": context.config["resource_group"], "APIM": apim.apim_resource_name,
        "gateway": apim.apim_resource_gateway_url,
        "backends": [b["backendId"] for b in context.config["llm_backends"]],
        "loaded models": list(dict.fromkeys(entry.name for entry in catalog.entries)),
        "models to test": [name for name in dict.fromkeys(entry.name for entry in catalog.entries)
                           if not catalog.is_skipped(name) and any(op in catalog.operations(name)
                           for op in ("chat", "embeddings", "responses"))],
    }, status="info", part="RESULT")

    model_names = list(dict.fromkeys(entry.name for entry in catalog.entries))
    utils.format_table("Loaded models and resolved operations", [{
        "model": name,
        "backends": ", ".join(dict.fromkeys(e.backend_id for e in catalog.entries if e.name == name)),
        "operations": catalog.operations(name),
        "source": ", ".join(dict.fromkeys(e.operations_source for e in catalog.entries if e.name == name)),
        "skipped": catalog.skip_reason(name),
    } for name in model_names])


model,backends,operations,source,skipped
gpt-4.1,aif-ttg2fgnpni24u-0,"[""chat"", ""responses""]",inferred,
gpt-5.2,"aif-ttg2fgnpni24u-0, aif-ttg2fgnpni24u-1","[""chat"", ""responses""]",inferred,
gpt-image-1.5,aif-ttg2fgnpni24u-0,"[""image-generation"", ""image-edit""]",inferred,
MAI-Image-2.5-Flash,aif-ttg2fgnpni24u-0,"[""image-generation"", ""image-edit""]",inferred,
FLUX.2-pro,aif-ttg2fgnpni24u-0,"[""image-generation"", ""image-edit""]",inferred,
text-embedding-3-large,"aif-ttg2fgnpni24u-0, aif-ttg2fgnpni24u-1","[""embeddings""]",inferred,
Mistral-Large-3,aif-ttg2fgnpni24u-0,"[""chat""]",inferred,
gpt-5.4-mini,"aif-ttg2fgnpni24u-0, aif-ttg2fgnpni24u-1","[""chat"", ""responses""]",inferred,


## 3. Inspect deployed resources and full proposed configuration

### 3. Inspect deployed resources and full proposed configuration

Run the visible 3. inspect deployed resources and full proposed configuration operations below; preserve their inspection output and validation evidence.

**Input:** `context`, `apim`

**Output:** `before`, `llm_backends_config`, `b`, `m`; visible result/evidence output.


In [5]:
with utils.notebook_step('3. Inspect deployed resources and full proposed configuration', inputs={'context': globals().get('context'), 'apim': globals().get('apim')}, expected='`before`, `llm_backends_config`, `b`, `m`; visible result/evidence output.', namespace=globals()):
    before = list(apim.client.backend.list_by_service(context.config["resource_group"], apim.apim_resource_name))
    utils.format_output('3. Inspect deployed resources and full proposed configuration', {'value_1': "Existing backend IDs:", 'value_2': [b.name for b in before]}, status='info', part='RESULT')
    llm_backends_config = context.config["llm_backends"]
    if not llm_backends_config:
        raise RuntimeError("Provide a complete LLM_BACKEND_CONFIG; partial discovery cannot reconstruct routing.")
    utils.format_output('3. Inspect deployed resources and full proposed configuration', {'value_1': "Proposed backend IDs:", 'value_2': [b["backendId"] for b in llm_backends_config]}, status='info', part='RESULT')
    utils.format_output('3. Inspect deployed resources and full proposed configuration', {'value_1': "Model names:", 'value_2': [m["name"] if isinstance(m, dict) else m
                           for b in llm_backends_config for m in b["supportedModels"]]}, status='info', part='RESULT')


## 4. Preview backend `.bicepparam`
Preserve authentication, model metadata, provider paths, circuit breakers and affinity settings. Supply all existing aliases; a partial update can overwrite shared policy fragments.

### 4. Preview backend .bicepparam

Run the visible 4. preview backend .bicepparam operations below; preserve their inspection output and validation evidence.



**Input:** `complete_backend_configuration`, `apim_coordinates`, `llm_backends_config`, `model_aliases`, `apim`, `run_dir`

**Output:** `identity`, `backend_parameters`, `backend_params`; visible result/evidence output.


In [6]:
with utils.notebook_step('4. Preview backend .bicepparam', inputs={'complete_backend_configuration': globals().get('complete_backend_configuration'), 'apim_coordinates': globals().get('apim_coordinates'), 'llm_backends_config': globals().get('llm_backends_config'), 'model_aliases': globals().get('model_aliases'), 'apim': globals().get('apim'), 'run_dir': globals().get('run_dir')}, expected='`identity`, `backend_parameters`, `backend_params`; visible result/evidence output.', namespace=globals()):
    identity = apim.get_managed_identity_info()
    if not identity.get("name"):
        raise RuntimeError("Specify a resolvable APIM user-assigned managed identity before onboarding.")
    backend_parameters = {
        "apim": apim_coordinates,
        "apimManagedIdentity": {"subscriptionId": context.config["subscription_id"],
            "resourceGroupName": identity["resourceGroup"], "name": identity["name"]},
        "llmBackendConfig": llm_backends_config, "modelAliases": model_aliases,
        "configureCircuitBreaker": True, "configureSessionAffinity": True,
    }
    backend_params = write_parameters(root / "bicep" / "infra" / "llm-backend-onboarding" / "main.bicep",
                                      backend_parameters, run_dir / "backends")
    utils.format_output('4. Preview backend .bicepparam', {'value_1': backend_params.read_text(encoding="utf-8")}, status='info', part='RESULT')


## 5. Deploy backend contracts

### 5. Deploy backend contracts

Run the visible 5. deploy backend contracts operations below; preserve their inspection output and validation evidence.

Run All deploys the complete backend configuration loaded from the selected azd environment. The target and parameter preview are shown first; no additional approval parameters are required.

**Input:** `backend_params`, `location`, `b`, `llm_backends_config`, `run`, `run_id`

**Output:** `backend_outputs`, `expected_ids`, `actual_ids`, `fragment`, `value`; visible result/evidence output.


In [7]:
with utils.notebook_step('5. Deploy backend contracts', inputs={'backend_params': globals().get('backend_params'), 'location': globals().get('location'), 'b': globals().get('b'), 'llm_backends_config': globals().get('llm_backends_config'), 'run': globals().get('run'), 'run_id': globals().get('run_id')}, expected='`backend_outputs`, `expected_ids`, `actual_ids`, `fragment`, `value`; visible result/evidence output.', namespace=globals()):
    backend_outputs = deploy_contract(backend_params, location, scoped_deployment_name("validation-backend-onboarding", apim.apim_service_id))
    expected_ids = {b["backendId"] for b in llm_backends_config}
    actual_ids = {b.name for b in apim.client.backend.list_by_service(
        context.config["resource_group"], apim.apim_resource_name)}
    run.check("backend-resources", expected_ids.issubset(actual_ids), expected=sorted(expected_ids), actual=sorted(actual_ids))
    for fragment in ("get-available-models", "set-backend-pools", "resolve-model-alias", "metadata-config"):
        value = apim.client.policy_fragment.get(context.config["resource_group"], apim.apim_resource_name, fragment).value
        run.check(f"fragment:{fragment}", bool(value), expected="Deployed policy XML")


## 6. Preview a temporary access contract

### 6. Preview a temporary access contract

Run the visible 6. preview a temporary access contract operations below; preserve their inspection output and validation evidence.

**Input:** `apim_coordinates`, `run_id`, `run_dir`, `scenario`

**Output:** `product_id`, `policy_xml`, `parameters`, `access_params`, `business_unit`; visible result/evidence output.


In [8]:
with utils.notebook_step('6. Preview a temporary access contract', inputs={'apim_coordinates': globals().get('apim_coordinates'), 'run_id': globals().get('run_id'), 'run_dir': globals().get('run_dir'), 'scenario': globals().get('scenario')}, expected='`product_id`, `policy_xml`, `parameters`, `access_params`, `business_unit`; visible result/evidence output.', namespace=globals()):
    business_unit, use_case, contract_environment = "Validation", scenario.replace("-", "")[:24], "DEV"
    product_id = f"LLM-{business_unit}-{use_case}-{contract_environment}"
    policy_xml = access_policy([], 100000, 1000000, "Hourly")
    parameters = {
        "apim": apim_coordinates, "keyVault": apim_coordinates,
        "useTargetAzureKeyVault": False,
        "useCase": {"businessUnit": business_unit, "useCaseName": use_case, "environment": contract_environment},
        "apiNameMapping": {"LLM": ["universal-llm-api"]},
        "services": [{"code": "LLM", "apiKeySecretName": product_id + "-KEY",
                      "endpointSecretName": product_id + "-ENDPOINT", "policyXml": policy_xml}],
    }
    access_params = write_parameters(root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
                                     parameters, run_dir / "access")
    utils.format_output('6. Preview a temporary access contract', {'policy_xml': policy_xml}, status='info', part='RESULT')
    utils.format_output('6. Preview a temporary access contract', {'value_1': access_params.read_text(encoding="utf-8")}, status='info', part='RESULT')


## 7. Deploy the access contract

### 7. Deploy the access contract

Run the visible 7. deploy the access contract operations below; preserve their inspection output and validation evidence.

**Input:** `product_id`, `access_params`, `location`, `apim`, `run`, `scenario`

**Output:** `owned_product_id`, `access_outputs`, `deployed_endpoints`, `product_id`, `api_key`; visible result/evidence output.


In [9]:
with utils.notebook_step('7. Deploy the access contract', inputs={'product_id': globals().get('product_id'), 'access_params': globals().get('access_params'), 'location': globals().get('location'), 'apim': globals().get('apim'), 'run': globals().get('run'), 'scenario': globals().get('scenario')}, expected='`owned_product_id`, `access_outputs`, `deployed_endpoints`, `product_id`, `api_key`; visible result/evidence output.', namespace=globals()):
    expected_product_id = product_id
    product_preexists = product_exists(context, apim, product_id)
    owned_product_id = None
    access_outputs = deploy_contract(access_params, location, scoped_deployment_name(f"validation-{scenario}-access", apim.apim_service_id))
    deployed_endpoints = access_outputs.get("endpoints", {}).get("value", [])
    if len(deployed_endpoints) != 1 or deployed_endpoints[0].get("productId") != expected_product_id:
        raise RuntimeError("Expected exactly one endpoint for the configured product; review parameter edits.")
    product_id = deployed_endpoints[0]["productId"]
    if not product_preexists:
        owned_product_id = product_id
    api_key = apim.get_subscription_key(deployed_endpoints[0]["subscriptionName"])
    run.check("access-contract-deployed", bool(access_outputs), expected="Nonempty deployment outputs")


## 8. Validate discovery

### 8. Validate discovery

Run the visible 8. validate discovery operations below; preserve their inspection output and validation evidence.

**Input:** `m`, `run`, `apim`, `b`, `context`

**Output:** `models_api`, `headers`, `response`, `body`, `items`; visible result/evidence output.


In [10]:
with utils.notebook_step('8. Validate discovery', inputs={'m': globals().get('m'), 'run': globals().get('run'), 'apim': globals().get('apim'), 'b': globals().get('b'), 'context': globals().get('context')}, expected='`models_api`, `headers`, `response`, `body`, `items`; visible result/evidence output.', namespace=globals()):
    models_api = apim.discover_api_exact("models")["url"]
    headers = {"api-key": api_key, "Content-Type": "application/json"}
    response = utils.http_call(requests.get, f"{models_api}/models", headers=headers, timeout=30)
    response.raise_for_status()
    body = response.json()
    items = body.get("data", body.get("value"))
    if not isinstance(items, list) or not items:
        raise RuntimeError("Discovery returned no model entries.")
    if any(not isinstance(item, dict) or not isinstance(item.get("name") or item.get("id"), str)
           or not (item.get("name") or item.get("id")) for item in items):
        raise RuntimeError("Each discovery entry must contain a nonempty model name or ID.")
    discovered_models = [item.get("name") or item.get("id") for item in items]
    discovered_aliases = {item.get("name") or item.get("id") for item in items
                          if item.get("type") == "alias"
                          or item.get("properties", {}).get("model", {}).get("format") == "Alias"}
    configured_models = {m["name"] if isinstance(m, dict) else m
                         for b in context.config["llm_backends"] for m in b["supportedModels"]}
    utils.format_output('8. Validate discovery', {'value_1': "Configured but not discovered:", 'value_2': sorted(configured_models - set(discovered_models))}, status='info', part='RESULT')
    utils.format_output('8. Validate discovery', {'value_1': "Discovered but not configured:", 'value_2': sorted(set(discovered_models) - configured_models)}, status='info', part='RESULT')
    run.check("model-discovery", configured_models.issubset(discovered_models),
              expected="All configured models visible to unrestricted contract", actual=discovered_models)


### 8. Validate discovery — continuation 9

Run the visible 8. validate discovery — continuation 9 operations below; preserve their inspection output and validation evidence.

**Input:** `headers`, `run`, `apim`, `response`, `configured_models`

**Output:** `path`, `endpoint`, `response`, `entries`, `names`; visible result/evidence output.


In [11]:
with utils.notebook_step("8. Validate Universal LLM deployment discovery", inputs={"endpoint": models_api}, expected="All configured models returned by Universal LLM discovery.", namespace=globals()):
    response = utils.http_call(requests.get, f"{models_api}/deployments", headers=headers, timeout=30)
    entries = response.json().get("value", []) if response.status_code == 200 else []
    names = {entry.get("name") for entry in entries}
    run.check("deployments:models", response.status_code == 200 and configured_models.issubset(names),
              expected="All configured deployments through Universal LLM", actual=response.status_code)


## 9. Test capability-specific operations through Universal LLM

### 9. Test capability-specific operations through Universal LLM

Run the visible 9. test capability-specific operations through universal llm operations below; preserve their inspection output and validation evidence.

**Input:** `discovered_models`, `discovered_aliases`, `catalog`, `run`

**Output:** `test_plan`, `model`, `operations`, `selected`, `op`; visible result/evidence output.


In [12]:
with utils.notebook_step('9. Test capability-specific operations through Universal LLM', inputs={'discovered_models': globals().get('discovered_models'), 'discovered_aliases': globals().get('discovered_aliases'), 'catalog': globals().get('catalog'), 'run': globals().get('run')}, expected='`test_plan`, `model`, `operations`, `selected`, `op`; visible result/evidence output.', namespace=globals()):
    test_plan = []
    for model in discovered_models:
        if catalog.is_skipped(model):
            run.skip(f"model:{model}", catalog.skip_reason(model))
            continue
        if model in discovered_aliases:
            run.skip(f"alias:{model}", "Alias strategy and member routing are covered in Model Aliases.")
            utils.print_info(f"Skipping alias model: {model}")
            continue
        operations = catalog.operations(model)
        selected = [op for op in operations if op in ("chat", "embeddings", "responses")]
        if not operations:
            run.blocked(f"capabilities:{model}", "Set VALIDATION_MODEL_CAPABILITIES for this deployed model.")
            utils.print_info(f"Blocked model with no operations: {model}")
        elif not selected:
            run.skip(f"operations:{model}", "Non-text operation; use Image Models or its capability-specific notebook.")
            utils.print_info(f"Skipping model with non-text operations: {model}")
        else:
            test_plan.extend((model, op) for op in selected)
    utils.print_info(f"Planned model/operation pairs: {test_plan}")


### 9. Test capability-specific operations through Universal LLM — continuation 11

Run the visible 9. test capability-specific operations through universal llm — continuation 11 operations below; preserve their inspection output and validation evidence.

**Input:** `test_plan`, `headers`, `model`, `response`, `run`, `models_api`

**Output:** `session`, `model`, `operation`, `case`, `response`; visible result/evidence output.


In [13]:
with utils.notebook_step('9. Test capability-specific operations through Universal LLM — continuation 11', inputs={'test_plan': globals().get('test_plan'), 'headers': globals().get('headers'), 'model': globals().get('model'), 'response': globals().get('response'), 'run': globals().get('run'), 'models_api': globals().get('models_api')}, expected='`session`, `model`, `operation`, `case`, `response`; visible result/evidence output.', namespace=globals()):
    with requests.Session() as session:
        session.headers.update(headers)
        for model, operation in test_plan:
            case = f"{model}:{operation}"
            try:
                if operation == "chat":
                    response = utils.http_call(session.post, f"{models_api}/chat/completions",
                        json={"model": model, "messages": [{"role": "user", "content": "What is 2+2? Reply briefly."}]},
                        timeout=90)
                    response.raise_for_status()
                    data = response.json()
                    content = data.get("choices", [{}])[0].get("message", {}).get("content")
                    run.check(case, bool(content), expected="Nonempty chat content", actual=response.status_code)
                elif operation == "embeddings":
                    response = utils.http_call(session.post, f"{models_api}/embeddings",
                        json={"model": model, "input": "A short validation sentence.", "encoding_format": "float"}, timeout=90)
                    response.raise_for_status()
                    vector = response.json().get("data", [{}])[0].get("embedding")
                    run.check(case, isinstance(vector, list) and bool(vector)
                              and all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vector),
                              expected="Nonempty numeric embedding", actual=response.status_code)
                else:
                    response = utils.http_call(session.post, f"{models_api}/responses",
                        json={"model": model, "input": "What is 2+2? Reply briefly."}, timeout=90)
                    response.raise_for_status()
                    data = response.json()
                    response_id = data.get("id")
                    run.check(case + ":create", bool(response_id) and bool(data.get("output")),
                              expected="Response ID and generated output", actual=response.status_code)
            except (requests.RequestException, ValueError, KeyError, IndexError) as exc:
                run.check(case, False, expected="Successful operation with valid body", actual=str(exc))


## 10. Universal LLM SDK consumption and streaming

### 10. Universal LLM SDK consumption and streaming

Run the visible 10. azure openai deployment-path request and sdk consumption operations below; preserve their inspection output and validation evidence.

**Input:** `catalog`, `model_override`, `overrides`, `headers`, `run`, `apim`

**Output:** `model`, `openai_api`, `api_version`, `response`, `client`; visible result/evidence output.


In [14]:
with utils.notebook_step("10. Universal LLM SDK consumption and streaming", inputs={"endpoint": models_api, "model override": model_override}, expected="SDK chat content and completed streaming through Universal LLM.", namespace=globals()):
    if not catalog.models_for("chat") and model_override is None:
        run.skip("universal-sdk", "No eligible chat models after mapping and skip filters.")
    else:
        from openai import OpenAI
        model = catalog.select("chat", override=model_override)
        utils.format_output("Universal LLM SDK request", {
            "endpoint": f"{models_api}/chat/completions", "model": model,
        }, part="INFO")
        with OpenAI(base_url=models_api.rstrip("/") + "/", api_key=api_key,
                    default_headers={"api-key": api_key}, max_retries=0, timeout=90) as client:
            reply = client.chat.completions.create(model=model, messages=[{"role": "user", "content": "Say hello briefly."}])
            content = reply.choices[0].message.content if reply.choices else None
            run.check("universal-sdk", bool(content), expected="SDK content", actual=content)
            if model not in catalog.models_for("streaming"):
                run.skip("universal-sdk-streaming", "The selected model override does not include streaming.")
            else:
                utils.format_output("Universal LLM SDK streaming request", {
                    "endpoint": f"{models_api}/chat/completions", "model": model, "stream": True,
                }, part="INFO")
                stream = client.chat.completions.create(model=model, stream=True,
                    messages=[{"role": "user", "content": "Count to three."}])
                chunks, finished = [], False
                with stream:
                    for chunk in stream:
                        for choice in chunk.choices:
                            chunks.append(choice.delta.content or "")
                            finished = finished or choice.finish_reason is not None
                run.check("universal-sdk-streaming", bool("".join(chunks).strip()) and finished,
                          expected="Nonempty streaming content and finish reason", actual="".join(chunks))


## 11. Validate configured aliases
Aliases must have compatible declared capabilities. Inspect both alias and resolved-model headers.

### 11. Validate configured aliases

Run the visible 11. validate configured aliases operations below; preserve their inspection output and validation evidence.

**Input:** `model_aliases`, `catalog`, `models_api`, `headers`, `run`

**Output:** Alias and resolved-member headers validated through Universal LLM.


In [15]:
with utils.notebook_step('11. Validate configured aliases', inputs={'model_aliases': globals().get('model_aliases'), 'run': globals().get('run'), 'headers': globals().get('headers'), 'apim': globals().get('apim'), 'endpoint': globals().get('models_api')}, expected='`alias`, `response`; visible result/evidence output.', namespace=globals()):
    if not model_aliases:
        run.skip("alias-routing", "No aliases supplied; use Model Aliases for priority/weighted coverage.")
    for alias in model_aliases:
        if catalog.is_skipped(alias["name"]) or any(catalog.is_skipped(m) for m in alias.get("models", [])):
            run.skip(f"alias:{alias['name']}", "Alias or one of its members is excluded by model filters.")
            continue
        if not alias.get("models") or any("chat" not in catalog.operations(m) for m in alias["models"]):
            run.blocked(f"alias:{alias['name']}", "Every alias member requires compatible chat capability.")
            continue
        response = utils.http_call(requests.post, f"{models_api}/chat/completions", headers=headers,
            json={"model": alias["name"], "messages": [{"role": "user", "content": "Say hello."}]}, timeout=90)
        run.check(f"alias:{alias['name']}:models", response.status_code == 200
                  and response.headers.get("UAIG-Alias") == alias["name"]
                  and response.headers.get("UAIG-Resolved-Model") in alias["models"],
                  expected="Alias header and valid member through Universal LLM", actual=response.status_code)


## 12. Summary

### 12. Summary

Run the visible 12. summary operations below; preserve their inspection output and validation evidence.

**Input:** `run`

**Output:** Visible inspection output and any validation cases produced by this cell.


In [16]:
with utils.notebook_step('12. Summary', inputs={'run': globals().get('run')}, expected='Visible inspection output and any validation cases produced by this cell.', namespace=globals()):
    run.finish()


case,status,required,expected,actual
backend-resources,PASS,True,"[""aif-ttg2fgnpni24u-0"", ""aif-ttg2fgnpni24u-1""]","[""aif-ttg2fgnpni24u-0"", ""aif-ttg2fgnpni24u-1"", ""content-safety-backend"", ""gpt-52-ai-foundry-backend-pool"", ""gpt-54-mini-ai-foundry-backend-pool"", ""text-embedding-3-large-ai-foundry-backend-pool""]"
fragment:get-available-models,PASS,True,Deployed policy XML,
fragment:set-backend-pools,PASS,True,Deployed policy XML,
fragment:resolve-model-alias,PASS,True,Deployed policy XML,
fragment:metadata-config,PASS,True,Deployed policy XML,
access-contract-deployed,PASS,True,Nonempty deployment outputs,
model-discovery,PASS,True,All configured models visible to unrestricted contract,"[""gpt-4.1"", ""gpt-5.2"", ""gpt-image-1.5"", ""MAI-Image-2.5-Flash"", ""FLUX.2-pro"", ""text-embedding-3-large"", ""Mistral-Large-3"", ""gpt-5.4-mini""]"
deployments:models,PASS,True,All configured deployments through Universal LLM,200
operations:gpt-image-1.5,SKIP,False,,Non-text operation; use Image Models or its capability-specific notebook.
operations:MAI-Image-2.5-Flash,SKIP,False,,Non-text operation; use Image Models or its capability-specific notebook.


## 13. Cleanup
Only the new access product is deletable here; shared routing is retained.

### 13. Cleanup

Run the visible 13. cleanup operations below; preserve their inspection output and validation evidence.

**Input:** `cleanup_enabled`, `product_id`, `owned_product_id`, `apim`, `context`

**Output:** Visible inspection output and any validation cases produced by this cell.


In [17]:
with utils.notebook_step('13. Cleanup', inputs={'cleanup_enabled': globals().get('cleanup_enabled'), 'product_id': globals().get('product_id'), 'owned_product_id': globals().get('owned_product_id'), 'apim': globals().get('apim'), 'context': globals().get('context')}, expected='Visible inspection output and any validation cases produced by this cell.', namespace=globals()):
    utils.format_output('13. Cleanup', {'value_1': "Owned product:", 'product_id': product_id}, status='info', part='RESULT')
    if cleanup_enabled and not product_preexists:
        if product_id != owned_product_id:
            raise RuntimeError("Edited parameters changed the product ID; ownership is unproven, so cleanup is blocked.")
        apim.client.product.delete(context.config["resource_group"], apim.apim_resource_name,
                                   product_id, delete_subscriptions=True, if_match="*")
        utils.format_output('13. Cleanup', {'value_1': "Deleted this run's access product and subscriptions. Shared backends were retained."}, status='info', part='RESULT')
    else:
        utils.format_output('13. Cleanup', {'value_1': "Cleanup disabled or product reused; existing resources and shared backends are retained."}, status='info', part='RESULT')
